# Critical structure in the unity family

This notebook is the slower companion to `reports/critical-structure.md`.

The narrow question is the useful one: **why does Newton iteration for `z^n - 1` get especially fragile near the center of the square, and why does that effect get worse as `n` grows?**


## 1. Start from the algebra

For

$$f(z) = z^n - 1,$$

Newton's method gives

$$N_n(z) = z - rac{z^n - 1}{n z^{n-1}} = rac{n-1}{n} z + rac{1}{n z^{n-1}}.$$

That formula already explains the main danger.
The origin is not a root. It is the point where the derivative vanishes, and the inverse-power term explodes there.

So the critical story here is not mysterious:

- the roots live on the unit circle
- the center is where the derivative collapses
- Newton's update becomes violent near that singularity


In [ ]:
from newton_fractal_lab.core import iterate_unity, scan_radius_bands

for power in [4, 8, 12]:
    near_origin = iterate_unity(complex(0.1, 0.1), power)
    near_root = iterate_unity(complex(0.9, 0.1), power)
    print(power, near_origin.iterations, near_origin.converged, near_origin.residual)
    print(power, near_root.iterations, near_root.converged, near_root.residual)


Even before looking at a whole grid, one local check makes the point: starts near the origin can burn the whole iteration budget while starts near an actual root settle quickly.


## 2. Radius bands give a cleaner family-level view

A basin image shows **where** the boundaries are, but it does not summarize the critical region very cleanly.
The radial scan bins points by starting radius and measures:

- mean iteration count
- converged fraction
- stalled fraction

That is not a full dynamical invariant. It is just a useful public bridge between the algebra and the rendered basins.


In [ ]:
profiles = {power: scan_radius_bands(power, width=120, height=120, max_iter=40, bands=12) for power in [3, 6, 9, 12]}
for power, rows in profiles.items():
    hardest = max(rows, key=lambda row: row.mean_iterations)
    weakest = min(rows, key=lambda row: row.converged_fraction)
    print(power, 'hardest', (round(hardest.radius_min, 2), round(hardest.radius_max, 2), round(hardest.mean_iterations, 2)))
    print(power, 'weakest', (round(weakest.radius_min, 2), round(weakest.radius_max, 2), round(weakest.converged_fraction, 3)))


## 3. What the radial profiles show

Three useful readings usually appear:

1. the inner bands are the hardest because they sit closest to the derivative singularity
2. higher powers keep that inner slow zone alive for longer
3. the outer square is not perfectly easy, but it is usually calmer than the center on the same iteration budget

That is a more precise claim than saying the fractal boundary is complicated.
The radius scan says *where the algebraic danger starts* and how it scales across the family.


## 4. Why the unit circle still matters

The roots themselves live on `|z| = 1`, so the unit circle is the natural radius marker in the figure.
But the difficult region is not just a thin ring at `|z| = 1`.
For this family, the deeper issue is the center term `1 / (n z^{n-1})`, which can fling iterates outward before they have any chance to settle into one basin.


## 5. Caveats

This notebook is intentionally modest.

- the radius bands depend on the chosen square `[-1.6, 1.6]^2`
- the measurements depend on the finite iteration budget
- the profiles do not replace a true critical-orbit or Julia-set analysis
- they are summaries of sampled starts, not proofs about every point on a circle


## 6. Problems and references

Try these next:

1. rerun the radius scan at two iteration budgets and ask which bands move the most
2. compare the radius profiles for one asymmetric polynomial against the unity family
3. check whether the slowest band always contains the origin-centered disk for the current square

Useful references to keep in mind:

- any standard complex dynamics text covering Newton maps and basins of attraction
- the repo's own `reports/unity-power-scan.md` and `reports/critical-structure.md` for the generated public summary
- the rendered artifact `art/critical-radius-scan.svg` for the fast visual pass
